# Gold — dimensões, fato e features do modelo

Lê a **silver** (`data/silver/`) e grava a **gold** em `data/gold/`. Como a silver, é função determinística da camada anterior: rodar duas vezes dá o mesmo resultado (seção 10 prova).

| Tabela | Grão (PK) | Papel |
| --- | --- | --- |
| `gold_dim_tempo` | mês | calendário: dias úteis, sazonalidade (sen/cos), tendência, dummy COVID |
| `gold_dim_enso` | mês | ONI, fase/episódio (**retrospectivos, só descritivos**) e indicadores **causais** |
| `gold_dim_uf` | UF | `dim_uf` + subsistema predominante + pesos **fixos** de consumo |
| `gold_fato_consumo_uf_mes` | UF × mês × classe | consumo cativo+livre somado, por dia, por consumidor |
| `gold_climatologia_uf` | UF × mês do ano | normais climatológicas do período-base |
| `gold_clima_uf_mes` | UF × mês | clima + anomalias vs. normais |
| `gold_clima_agregado_mes` | escopo (região/Brasil) × mês | clima agregado com pesos fixos |
| `gold_features_uf_mes` | UF × mês × classe | tabela do modelo: alvos, controles, clima, ONI defasado, autorregressivas |

Decisões que valem para tudo abaixo:

- **Janela derivada da silver**, nunca fixada aqui. Se a silver for estendida, a gold acompanha.
- **Período-base fixo** (`BASE_ANOS`) para normais climatológicas e pesos regionais. Como cobre os primeiros anos da série, fica dentro do treino de qualquer divisão temporal cronológica.
- **Defasagens por data, não por posição**: um mês faltando vira nulo, não puxa o valor do mês errado.
- **ONI**: o ONI de um mês usa o mês seguinte. Lag 1 não enxerga o futuro do mês-alvo, mas só é conhecido ao fim dele. **Para prever antes do mês começar, use lag ≥ 2.**
- **`fase_enso` da silver é retrospectiva** (um episódio só existe com 5 meses seguidos, contando os futuros). Fica em `gold_dim_enso` para descrição e gráficos. As features usam apenas `el_nino_causal`/`la_nina_causal`, que só olham o passado.
- **Sem divisão treino/teste** aqui: ela pertence à etapa de modelagem.
- **Papéis explícitos** em `gold_features_uf_mes` (coluna `papel` do dicionário): `chave`, `alvo` ou `feature`. Nenhuma feature usa o consumo do próprio mês: consumo, consumidores e `pct_livre` do mês ficam só no fato; nas features entram defasados. `testar_sem_vazamento` prova isso a cada execução.

In [ ]:
import contextlib
import hashlib
import io
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.easter import easter

RAIZ = Path.cwd().parent
DATA = RAIZ / "data"
SILVER = DATA / "silver"
GOLD = DATA / "gold"
RELATORIOS = GOLD / "_relatorios"

# Período-base: normais climatológicas e pesos fixos de agregação. Precisa caber na janela da silver.
BASE_ANOS = (2004, 2014)

# ENSO (critério NOAA): ONI >= 0.5 (El Niño) ou <= -0.5 (La Niña) por 5 meses seguidos.
LIMIAR_ENSO = 0.5
MESES_EPISODIO = 5

# Defasagens (em meses, relativas ao mês-alvo). ONI lag 1 só é conhecido ao fim do mês-alvo;
# para previsão antes do mês começar, use lag >= 2.
LAGS_ONI = (1, 2, 3, 4, 5, 6)
LAGS_FLAG = (1, 2)

# Dummy COVID: premissa documentada, ajuste se o grupo adotar outro recorte.
COVID_INI = pd.Timestamp("2020-03-01")
COVID_FIM = pd.Timestamp("2021-06-01")

## 1. Esquemas e dicionário

Um dicionário único `COLUNAS` (nome → tipo, descrição) alimenta `aplicar_schema` e o `dicionario_gold.csv`, que traz também o domínio de valores válidos, a fonte, o URL, a licença e a data de coleta (tirada das cargas da silver usadas, `_id_carga`). O mesmo nome tem o mesmo significado em todas as tabelas.

In [ ]:
S, I8, I16, I64, F, D, B = "string", "Int8", "Int16", "Int64", "float64", "datetime64[ns]", "boolean"

COLUNAS = {
    # --- chaves e dimensões
    "cod_ibge_uf": (S, "Código IBGE da UF (2 dígitos, texto)"),
    "uf": (S, "Sigla da UF"),
    "nome_uf": (S, "Nome da UF"),
    "regiao": (S, "Região geográfica: norte, nordeste, centro_oeste, sudeste, sul"),
    "capital_proxy_clima": (S, "Capital cujo clima representa a UF"),
    "data_ref": (D, "Mês de referência (sempre dia 1)"),
    "ano": (I16, "Ano de data_ref"),
    "mes": (I8, "Mês (1-12) de data_ref"),
    "classe": (S, "Classe de consumo (EPE)"),
    "escopo": (S, "Nível de agregação: regiao ou brasil"),
    "id_escopo": (S, "Região ou 'brasil'"),
    # --- calendário
    "trimestre": (I8, "Trimestre civil"),
    "dias_no_mes": (I8, "Dias corridos do mês"),
    "dias_uteis": (I8, "Dias úteis: seg-sex menos feriados nacionais (sem feriados estaduais/municipais, Carnaval, Corpus Christi)"),
    "mes_sin": (F, "sen(2π·mes/12): sazonalidade contínua"),
    "mes_cos": (F, "cos(2π·mes/12): sazonalidade contínua"),
    "t": (I16, "Tendência: meses desde o início da janela (0 = primeiro mês)"),
    "flag_covid": (B, "Mês dentro do recorte COVID (COVID_INI..COVID_FIM)"),
    # --- ENSO
    "oni": (F, "Oceanic Niño Index (média móvel centrada de 3 meses: usa o mês seguinte)"),
    "fase_enso": (S, "RETROSPECTIVA (usa meses futuros): el_nino/la_nina/neutro. Não usar como feature"),
    "intensidade_enso": (S, "RETROSPECTIVA: intensidade do episódio. Não usar como feature"),
    "episodio_id": (I16, "RETROSPECTIVO: sequência do episódio El Niño/La Niña (nulo no neutro)"),
    "mes_no_episodio": (I16, "RETROSPECTIVO: mês dentro do episódio (1..n)"),
    "meses_consec_quente": (I16, "Meses seguidos até t com ONI >= 0.5 (só passado)"),
    "meses_consec_frio": (I16, "Meses seguidos até t com ONI <= -0.5 (só passado)"),
    "el_nino_causal": (B, "meses_consec_quente >= 5: El Niño já confirmado em t (só passado)"),
    "la_nina_causal": (B, "meses_consec_frio >= 5: La Niña já confirmada em t (só passado)"),
    # --- pesos
    "subsistema_predominante": (S, "Subsistema com maior consumo da UF no período-base"),
    "n_subsistemas": (I8, "Subsistemas distintos em que a UF aparece (>1: ex. SIN + isolado)"),
    "consumo_base_mwh": (F, "Consumo total da UF no período-base"),
    "peso_br": (F, "Participação fixa da UF no consumo do Brasil (período-base); soma 1"),
    "peso_regiao": (F, "Participação fixa da UF no consumo da região (período-base); soma 1 por região"),
    # --- consumo
    "consumo_mwh": (F, "Consumo faturado, cativo + livre, todos os subsistemas da UF (unidade a confirmar na EPE; MWh)"),
    "consumo_mwh_cativo": (F, "Parcela cativa"),
    "consumo_mwh_livre": (F, "Parcela livre (pode ter estornos negativos)"),
    "n_consumidores": (I64, "Consumidores (cativo + livre)"),
    "consumo_mwh_por_dia": (F, "consumo_mwh / dias_no_mes (neutraliza o tamanho do mês)"),
    "consumo_mwh_por_consumidor": (F, "consumo_mwh / n_consumidores"),
    "pct_livre": (F, "Parcela livre / consumo total (nulo se consumo <= 0; pode sair de [0,1] por estornos)"),
    "tem_estorno": (B, "Alguma linha da silver neste grão tem consumo negativo"),
    "flag_consumo_nao_positivo": (B, "consumo_mwh <= 0 após a soma (alvo em log fica nulo)"),
    # --- clima
    "temp_media_c": (F, "Temperatura média do mês (°C)"),
    "temp_max_media_c": (F, "Média das máximas diárias (°C)"),
    "temp_max_abs_c": (F, "Máxima absoluta do mês (°C)"),
    "precip_total_mm": (F, "Precipitação total do mês (mm)"),
    "dias_chuva": (I16, "Dias com chuva (limiar da silver)"),
    "dias_calor": (I16, "Dias com Tmax acima do limiar da silver (35 °C): pode ser quase sempre 0"),
    "dias_chuva_pond": (F, "dias_chuva médio ponderado pelos pesos fixos"),
    "dias_calor_pond": (F, "dias_calor médio ponderado pelos pesos fixos"),
    "clim_temp_media_c": (F, "Normal (média no período-base) da temperatura média, por UF e mês do ano"),
    "clim_temp_max_media_c": (F, "Normal da média das máximas"),
    "clim_precip_total_mm": (F, "Normal da precipitação total"),
    "clim_dias_calor": (F, "Normal de dias_calor"),
    "n_anos_base": (I8, "Anos do período-base usados na normal"),
    "anom_temp_media_c": (F, "temp_media_c - normal (°C)"),
    "anom_temp_max_media_c": (F, "temp_max_media_c - normal (°C)"),
    "anom_precip_mm": (F, "precip_total_mm - normal (mm)"),
    "anom_dias_calor": (F, "dias_calor - normal"),
    # --- alvos do modelo
    "y_log_consumo_dia": (F, "ln(consumo_mwh_por_dia); nulo se consumo <= 0"),
    "y_yoy_log_consumo_dia": (F, "y_log_consumo_dia - mesmo mês do ano anterior (remove sazonalidade e boa parte da tendência)"),
    # --- autorregressivas
    "ar_y_log_lag1": (F, "y_log_consumo_dia do mês anterior (exige horizonte de 1 mês)"),
    "ar_y_log_lag12": (F, "y_log_consumo_dia de 12 meses antes"),
    "pct_livre_lag1": (F, "pct_livre do mês anterior (o do próprio mês sai do consumo-alvo)"),
}
for _k in LAGS_ONI:
    COLUNAS[f"oni_lag{_k}"] = (F, f"ONI de {_k} mês(es) antes. Lag 1 só é conhecido ao fim do mês-alvo; lag >= 2 antes de ele começar")
for _k in LAGS_FLAG:
    COLUNAS[f"el_nino_causal_lag{_k}"] = (B, f"el_nino_causal de {_k} mês(es) antes")
    COLUNAS[f"la_nina_causal_lag{_k}"] = (B, f"la_nina_causal de {_k} mês(es) antes")
for _c in ("temp_media_c", "temp_max_media_c", "precip_total_mm", "anom_temp_media_c", "anom_temp_max_media_c", "anom_precip_mm"):
    COLUNAS[f"{_c}_lag1"] = (F, f"{_c} do mês anterior (o faturamento defasa o consumo em ~1 mês)")

COLS_CLIMA_NIVEL = ["temp_media_c", "temp_max_media_c", "precip_total_mm", "dias_chuva", "dias_calor",
                    "anom_temp_media_c", "anom_temp_max_media_c", "anom_precip_mm", "anom_dias_calor"]
COLS_CLIMA_LAG1 = ["temp_media_c", "temp_max_media_c", "precip_total_mm",
                   "anom_temp_media_c", "anom_temp_max_media_c", "anom_precip_mm"]
COLS_ONI_LAGS = [f"oni_lag{k}" for k in LAGS_ONI]
COLS_FLAG_LAGS = [f"{p}_lag{k}" for k in LAGS_FLAG for p in ("el_nino_causal", "la_nina_causal")]

# Papéis em gold_features_uf_mes: o que não é chave nem alvo é feature. Consumo, consumidores e
# pct_livre do próprio mês ficam só no fato (são o alvo ou derivam dele).
COLS_CHAVE = ["cod_ibge_uf", "uf", "regiao", "data_ref", "classe"]
COLS_ALVO = ["y_log_consumo_dia", "y_yoy_log_consumo_dia"]

SCHEMAS = {
    "gold_dim_tempo": {"pk": ["data_ref"], "colunas": [
        "data_ref", "ano", "mes", "trimestre", "dias_no_mes", "dias_uteis", "mes_sin", "mes_cos", "t", "flag_covid"]},
    "gold_dim_enso": {"pk": ["data_ref"], "colunas": [
        "data_ref", "oni", "fase_enso", "intensidade_enso", "episodio_id", "mes_no_episodio",
        "meses_consec_quente", "meses_consec_frio", "el_nino_causal", "la_nina_causal"]},
    "gold_dim_uf": {"pk": ["cod_ibge_uf"], "colunas": [
        "cod_ibge_uf", "uf", "nome_uf", "regiao", "capital_proxy_clima", "subsistema_predominante",
        "n_subsistemas", "consumo_base_mwh", "peso_br", "peso_regiao"]},
    "gold_fato_consumo_uf_mes": {"pk": ["cod_ibge_uf", "data_ref", "classe"], "colunas": [
        "cod_ibge_uf", "uf", "regiao", "data_ref", "ano", "mes", "classe", "consumo_mwh", "consumo_mwh_cativo",
        "consumo_mwh_livre", "n_consumidores", "consumo_mwh_por_dia", "consumo_mwh_por_consumidor",
        "pct_livre", "tem_estorno", "flag_consumo_nao_positivo"]},
    "gold_climatologia_uf": {"pk": ["cod_ibge_uf", "mes"], "colunas": [
        "cod_ibge_uf", "uf", "mes", "clim_temp_media_c", "clim_temp_max_media_c", "clim_precip_total_mm",
        "clim_dias_calor", "n_anos_base"]},
    "gold_clima_uf_mes": {"pk": ["cod_ibge_uf", "data_ref"], "colunas": [
        "cod_ibge_uf", "uf", "data_ref", "ano", "mes", "temp_media_c", "temp_max_media_c", "temp_max_abs_c",
        "precip_total_mm", "dias_chuva", "dias_calor", "anom_temp_media_c", "anom_temp_max_media_c",
        "anom_precip_mm", "anom_dias_calor"]},
    "gold_clima_agregado_mes": {"pk": ["escopo", "id_escopo", "data_ref"], "colunas": [
        "escopo", "id_escopo", "data_ref", "ano", "mes", "temp_media_c", "temp_max_media_c", "precip_total_mm",
        "dias_chuva_pond", "dias_calor_pond", "anom_temp_media_c", "anom_temp_max_media_c", "anom_precip_mm",
        "anom_dias_calor"]},
    "gold_features_uf_mes": {"pk": ["cod_ibge_uf", "data_ref", "classe"], "colunas": (
        COLS_CHAVE + COLS_ALVO
        + ["ano", "mes", "pct_livre_lag1", "dias_no_mes", "dias_uteis", "mes_sin", "mes_cos", "t", "flag_covid"]
        + COLS_CLIMA_NIVEL + [f"{c}_lag1" for c in COLS_CLIMA_LAG1]
        + COLS_ONI_LAGS + COLS_FLAG_LAGS + ["ar_y_log_lag1", "ar_y_log_lag12"])},
}

VALIDACOES = []


def checar(nome, ok, detalhe="", critico=True):
    """Registra a validação; as críticas interrompem a execução."""
    VALIDACOES.append({"nome": nome, "ok": bool(ok), "critico": critico, "detalhe": str(detalhe)})
    print(("OK    " if ok else ("FALHA " if critico else "AVISO ")) + nome + (f" — {detalhe}" if detalhe else ""))
    assert ok or not critico, f"validação crítica falhou: {nome} — {detalhe}"


def aplicar_schema(df, tabela):
    """Seleciona e tipa as colunas da tabela, ordena pela PK e confere a chave."""
    sch = SCHEMAS[tabela]
    faltam = [c for c in sch["colunas"] if c not in df.columns]
    assert not faltam, f"{tabela}: colunas ausentes {faltam}"
    out = df[sch["colunas"]].copy()
    for c in sch["colunas"]:
        out[c] = out[c].astype(COLUNAS[c][0])
    out = out.sort_values(sch["pk"]).reset_index(drop=True)
    dup = int(out.duplicated(sch["pk"]).sum())
    checar(f"{tabela}: PK única {sch['pk']}", dup == 0, f"{dup} duplicatas")
    return out


def hash_tabela(df):
    """Hash do conteúdo, independente da ordem das linhas."""
    h = pd.util.hash_pandas_object(df.reset_index(drop=True), index=False)
    return hashlib.sha256(np.sort(h.to_numpy())).hexdigest()


FONTES = {  # chave: (instituição, URL, licença) — as mesmas da silver
    "epe": ("EPE — consumo mensal de energia elétrica",
            "https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/dados-abertos/Documents/Dados_abertos_Consumo_Mensal.xlsx",
            "CC BY 4.0"),
    "noaa": ("NOAA CPC/PSL — Oceanic Niño Index", "https://psl.noaa.gov/data/correlation/oni.data",
             "Domínio público (governo federal dos EUA)"),
    "openmeteo": ("Open-Meteo Archive API (reanálise ERA5, Copernicus C3S)", "https://archive-api.open-meteo.com/v1/archive",
                  "CC BY 4.0 (Open-Meteo); ERA5 sob a licença do Copernicus"),
}
ORIGEM = {  # tabela: fontes (vazio = gerada no código)
    "gold_dim_tempo": [], "gold_dim_enso": ["noaa"], "gold_dim_uf": ["epe"], "gold_fato_consumo_uf_mes": ["epe"],
    "gold_climatologia_uf": ["openmeteo"], "gold_clima_uf_mes": ["openmeteo"],
    "gold_clima_agregado_mes": ["openmeteo", "epe"], "gold_features_uf_mes": ["epe", "openmeteo", "noaa"],
}
BOOL, REAL_OU_NULO = "True, False", "real ou nulo"
DOMINIO = {  # colunas *_lagK herdam o domínio da coluna base
    "cod_ibge_uf": "27 códigos IBGE de UF", "uf": "27 siglas", "nome_uf": "27 nomes",
    "regiao": "norte, nordeste, centro_oeste, sudeste, sul", "capital_proxy_clima": "27 capitais",
    "data_ref": "dia 1 de cada mês da janela da silver (gold_dim_enso: série inteira do ONI)",
    "ano": "ano de data_ref", "mes": "1 a 12", "classe": "residencial, comercial, industrial, rural, outros",
    "escopo": "regiao, brasil", "id_escopo": "uma das 5 regiões ou brasil",
    "trimestre": "1 a 4", "dias_no_mes": "28 a 31", "dias_uteis": "0 a 23", "mes_sin": "-1 a 1", "mes_cos": "-1 a 1",
    "t": "inteiro >= 0", "flag_covid": BOOL,
    "oni": "-5 a 5 °C (anomalia)", "fase_enso": "el_nino, la_nina, neutro",
    "intensidade_enso": "fraca, moderada, forte, muito_forte ou nulo (neutro)",
    "episodio_id": "inteiro >= 1 ou nulo (neutro)", "mes_no_episodio": "inteiro >= 1 ou nulo (neutro)",
    "meses_consec_quente": "inteiro >= 0", "meses_consec_frio": "inteiro >= 0",
    "el_nino_causal": BOOL, "la_nina_causal": BOOL,
    "subsistema_predominante": "N, NE, SE_CO, S, ISOLADO", "n_subsistemas": "1 a 5", "consumo_base_mwh": "real > 0",
    "peso_br": "0 a 1; soma 1", "peso_regiao": "0 a 1; soma 1 por região",
    "consumo_mwh": "real (estornos podem deixar <= 0)", "consumo_mwh_cativo": "real", "consumo_mwh_livre": "real",
    "n_consumidores": "inteiro >= 0", "consumo_mwh_por_dia": "real",
    "consumo_mwh_por_consumidor": "real ou nulo (sem consumidores)",
    "pct_livre": "real ou nulo (consumo <= 0); normalmente 0 a 1, sai disso por estornos",
    "tem_estorno": BOOL, "flag_consumo_nao_positivo": BOOL,
    "temp_media_c": "-30 a 50 °C", "temp_max_media_c": "-30 a 55 °C", "temp_max_abs_c": "-30 a 55 °C",
    "precip_total_mm": "real >= 0", "dias_chuva": "0 a 31", "dias_calor": "0 a 31",
    "dias_chuva_pond": "0 a 31", "dias_calor_pond": "0 a 31",
    "clim_temp_media_c": "-30 a 50 °C", "clim_temp_max_media_c": "-30 a 55 °C", "clim_precip_total_mm": "real >= 0",
    "clim_dias_calor": "0 a 31", "n_anos_base": f"1 a {BASE_ANOS[1] - BASE_ANOS[0] + 1}",
    "anom_temp_media_c": "real", "anom_temp_max_media_c": "real", "anom_precip_mm": "real", "anom_dias_calor": "real",
    "y_log_consumo_dia": "real ou nulo (consumo <= 0)", "y_yoy_log_consumo_dia": REAL_OU_NULO + " (primeiros 12 meses)",
    "ar_y_log_lag1": REAL_OU_NULO + " (primeiro mês)", "ar_y_log_lag12": REAL_OU_NULO + " (primeiros 12 meses)",
}


def dominio(c):
    return DOMINIO.get(c) or DOMINIO.get(re.sub(r"_lag\d+$", "", c))


def data_coleta(*cargas):
    """Período das cargas usadas, em data (o id da carga é o instante UTC em que ela começou)."""
    ids = sorted(set(pd.concat(cargas).dropna().astype(str)))
    ini, fim = (f"{i[:4]}-{i[4:6]}-{i[6:8]}" for i in (ids[0], ids[-1]))
    return ini if ini == fim else f"{ini} a {fim}"


def dicionario(coleta):
    """coleta: {fonte: data de coleta}, calculada das cargas da silver em construir_gold."""
    faltam = [c for c in COLUNAS if not dominio(c)]
    assert not faltam, f"colunas sem domínio no dicionário: {faltam}"
    linhas = []
    for tabela, sch in SCHEMAS.items():
        for c in sch["colunas"]:
            papel = ("" if tabela != "gold_features_uf_mes" else
                     "chave" if c in COLS_CHAVE else "alvo" if c in COLS_ALVO else "feature")
            f = ORIGEM[tabela]
            linhas.append({"tabela": tabela, "coluna": c, "tipo": COLUNAS[c][0], "chave": c in sch["pk"],
                           "papel": papel, "dominio": dominio(c), "descricao": COLUNAS[c][1],
                           "fonte": "; ".join(FONTES[k][0] for k in f) or "gerada no código (calendário e feriados nacionais)",
                           "url": "; ".join(FONTES[k][1] for k in f), "licenca": "; ".join(FONTES[k][2] for k in f),
                           "data_coleta": "; ".join(coleta[k] for k in f)})
    return pd.DataFrame(linhas)

## 2. Funções de tempo, defasagem e agregação

`com_defasagem` casa por **data**: o valor de t-k entra na linha de t, e um mês ausente na fonte vira nulo (um `shift` posicional puxaria o mês errado).

In [ ]:
def adiantar(df, k):
    """Soma k meses a data_ref: o valor do mês t passa a valer em t+k."""
    out = df.copy()
    out["data_ref"] = (out["data_ref"].dt.to_period("M") + k).dt.to_timestamp().astype("datetime64[ns]")
    return out


def com_defasagem(alvo, fonte, mapa, k, chaves=()):
    """Traz para cada linha de `alvo` o valor de `fonte` k meses antes (casando por chaves + data).
    mapa: {coluna_na_fonte: nome_novo}."""
    chaves = list(chaves)
    f = adiantar(fonte[chaves + ["data_ref"] + list(mapa)], k).rename(columns=mapa)
    return alvo.merge(f, on=chaves + ["data_ref"], how="left", validate="m:1")


def feriados_nacionais(ano):
    """Feriados nacionais (dias de semana ou não; busday_count ignora fins de semana).
    Inclui Sexta-feira Santa; Consciência Negra só a partir de 2024 (Lei 14.759/2023)."""
    fixos = [(1, 1), (4, 21), (5, 1), (9, 7), (10, 12), (11, 2), (11, 15), (12, 25)]
    dias = [np.datetime64(f"{ano}-{m:02d}-{d:02d}") for m, d in fixos]
    dias.append(np.datetime64(easter(ano)) - np.timedelta64(2, "D"))
    if ano >= 2024:
        dias.append(np.datetime64(f"{ano}-11-20"))
    return dias


def dias_uteis_mes(inicio):
    fim = inicio + pd.offsets.MonthBegin(1)
    feriados = np.array(feriados_nacionais(inicio.year), dtype="datetime64[D]")
    return int(np.busday_count(np.datetime64(inicio.date()), np.datetime64(fim.date()), holidays=feriados))


def corridas(cond):
    """Meses seguidos, até t inclusive, em que `cond` vale True (0 onde vale False). Só olha o passado."""
    cond = cond.astype(bool)
    bloco = (cond != cond.shift()).cumsum()
    return cond.astype(int).groupby(bloco).cumsum()


def flags_causais(oni):
    """Indicadores de ENSO que só usam o ONI até t (série contígua, em ordem). Um episódio só é
    'confirmado' no 5º mês seguido, então o indicador atrasa o início real em 4 meses: é o preço de
    não usar o futuro."""
    oni = pd.Series(np.asarray(oni, dtype="float64"))
    quente = corridas(oni >= LIMIAR_ENSO)
    frio = corridas(oni <= -LIMIAR_ENSO)
    return pd.DataFrame({
        "meses_consec_quente": quente, "meses_consec_frio": frio,
        "el_nino_causal": quente >= MESES_EPISODIO, "la_nina_causal": frio >= MESES_EPISODIO})


def media_ponderada(df, grupo, peso, colunas):
    """Média ponderada por (grupo, data_ref): soma(peso·x) / soma(peso)."""
    tmp = df[[grupo, "data_ref"]].copy()
    w = df[peso].astype("float64")
    for c in colunas:
        tmp[c] = df[c].astype("float64") * w
    tmp["_w"] = w
    s = tmp.groupby([grupo, "data_ref"], as_index=False).sum()
    for c in colunas:
        s[c] = s[c] / s["_w"]
    return s.drop(columns="_w")

## 3. Testes das funções

Rodam a cada execução, antes de construir qualquer tabela, com dados sintéticos de resposta conhecida.

In [ ]:
def _meses(*datas):
    return pd.to_datetime(list(datas)).astype("datetime64[ns]")


def testar_defasagem_por_data():
    fonte = pd.DataFrame({"data_ref": _meses("2020-01-01", "2020-02-01", "2020-04-01"), "v": [1.0, 2.0, 4.0]})
    alvo = pd.DataFrame({"data_ref": _meses("2020-02-01", "2020-03-01", "2020-04-01", "2020-05-01")})
    r1 = com_defasagem(alvo, fonte, {"v": "v1"}, 1)
    assert np.allclose(r1["v1"], [1, 2, np.nan, 4], equal_nan=True), r1  # mês 03 ausente na fonte → nulo
    r2 = com_defasagem(alvo, fonte, {"v": "v2"}, 2)
    assert np.allclose(r2["v2"], [np.nan, 1, 2, np.nan], equal_nan=True), r2
    # com chave: UFs não se misturam
    fonte2 = pd.concat([fonte.assign(g="A"), fonte.assign(g="B", v=fonte["v"] * 10)], ignore_index=True)
    alvo2 = pd.concat([alvo.assign(g="A"), alvo.assign(g="B")], ignore_index=True)
    r3 = com_defasagem(alvo2, fonte2, {"v": "v1"}, 1, ["g"])
    assert np.allclose(r3["v1"], [1, 2, np.nan, 4, 10, 20, np.nan, 40], equal_nan=True), r3
    checar("teste: defasagem casa por data (mês ausente vira nulo, chaves não se misturam)", True)


def testar_flags_causais():
    const = flags_causais([0.6] * 7)
    assert list(const["el_nino_causal"]) == [False] * 4 + [True] * 3, const  # confirma no 5º mês
    assert not const["la_nina_causal"].any()
    rng = np.random.default_rng(0)
    oni = np.sin(np.arange(160) / 9) * 1.2 + rng.normal(0, 0.15, 160)
    cheio = flags_causais(oni)
    assert cheio["el_nino_causal"].any() and cheio["la_nina_causal"].any() and not cheio["el_nino_causal"].all()
    for t in (30, 55, 80, 120, 159):  # o valor em t não pode mudar quando o futuro é cortado
        corte = flags_causais(oni[: t + 1])
        assert (cheio.iloc[: t + 1].to_numpy() == corte.to_numpy()).all(), f"flags dependem do futuro em t={t}"
    checar("teste: flags de ENSO são causais (cortar o futuro não muda o passado)", True)


def testar_dias_uteis():
    esperado = {"2024-01-01": 22,   # 23 dias de semana − Confraternização
                "2024-03-01": 20,   # 21 − Sexta-feira Santa (29/03)
                "2023-11-01": 20,   # 22 − Finados − República (20/11 ainda não era feriado)
                "2024-11-01": 19}   # 21 − República − Consciência Negra (nova em 2024)
    for mes, n in esperado.items():
        assert dias_uteis_mes(pd.Timestamp(mes)) == n, (mes, dias_uteis_mes(pd.Timestamp(mes)), n)
    checar("teste: dias úteis (feriados fixos, Sexta-feira Santa, Consciência Negra desde 2024)", True)

## 4. Leitura da silver e conferências de entrada

A janela vem da própria silver. Se o clima não cobrir todo UF-mês da energia (por exemplo, a silver foi estendida mas o Open-Meteo não foi reingerido), a gold **para** com a instrução do que fazer, em vez de encolher a janela em silêncio.

In [ ]:
TABELAS_SILVER = ["dim_uf", "silver_energia_uf", "silver_clima_uf_mes", "silver_oni_mes"]


def ler_silver():
    faltam = [n for n in TABELAS_SILVER if not (SILVER / f"{n}.parquet").exists()]
    assert not faltam, f"silver ausente: {faltam}. Rode notebooks/silver.ipynb antes."
    return {n: pd.read_parquet(SILVER / f"{n}.parquet") for n in TABELAS_SILVER}


def conferir_entrada(s):
    energia, clima, oni = s["silver_energia_uf"], s["silver_clima_uf_mes"], s["silver_oni_mes"]
    ini, fim = energia["data_ref"].min(), energia["data_ref"].max()
    meses = pd.date_range(ini, fim, freq="MS")
    checar("janela: meses da energia contíguos", energia["data_ref"].nunique() == len(meses),
           f"{ini:%Y-%m} a {fim:%Y-%m} ({len(meses)} meses)")

    chaves = ["cod_ibge_uf", "data_ref"]
    m = energia[chaves].drop_duplicates().merge(clima[chaves], on=chaves, how="left", indicator=True)
    sem_clima = int((m["_merge"].astype(str) == "left_only").sum())
    checar("clima cobre todo UF-mês da energia", sem_clima == 0,
           f"{sem_clima} UF-meses sem clima. Se a janela da silver foi estendida, reingira o Open-Meteo "
           "(ingestao.ipynb) e rode a silver de novo" if sem_clima else "")

    checar("ONI cobre a janela e as defasagens",
           oni["data_ref"].min() <= ini - pd.DateOffset(months=max(LAGS_ONI)) and oni["data_ref"].max() >= fim,
           f"ONI {oni['data_ref'].min():%Y-%m} a {oni['data_ref'].max():%Y-%m}")
    checar("período-base cabe na janela", ini.year <= BASE_ANOS[0] and fim.year >= BASE_ANOS[1],
           f"BASE_ANOS={BASE_ANOS}, janela {ini.year}-{fim.year}")
    checar("energia: 27 UFs", energia["cod_ibge_uf"].nunique() == 27, f"{energia['cod_ibge_uf'].nunique()} UFs")
    return ini, fim

## 5. Dimensões

`gold_dim_enso` guarda duas famílias de colunas que **não se misturam**: as retrospectivas (`fase_enso`, `intensidade_enso`, `episodio_id`, `mes_no_episodio`), boas para gráficos e estudo de eventos, e as causais (`meses_consec_*`, `*_causal`), que são as únicas que viram feature.

Os pesos de `gold_dim_uf` usam o consumo do **período-base**, não o do próprio mês: pesos que se movem com o consumo contaminariam a análise com o próprio alvo.

In [ ]:
def montar_dim_tempo(ini, fim):
    d = pd.DataFrame({"data_ref": pd.date_range(ini, fim, freq="MS")})
    d["ano"] = d["data_ref"].dt.year
    d["mes"] = d["data_ref"].dt.month
    d["trimestre"] = d["data_ref"].dt.quarter
    d["dias_no_mes"] = d["data_ref"].dt.days_in_month
    d["dias_uteis"] = [dias_uteis_mes(x) for x in d["data_ref"]]
    d["mes_sin"] = np.sin(2 * np.pi * d["mes"] / 12)
    d["mes_cos"] = np.cos(2 * np.pi * d["mes"] / 12)
    d["t"] = np.arange(len(d))
    d["flag_covid"] = d["data_ref"].between(COVID_INI, COVID_FIM)
    return aplicar_schema(d, "gold_dim_tempo")


def montar_dim_enso(oni):
    o = oni.sort_values("data_ref").reset_index(drop=True)
    contiguo = len(pd.date_range(o["data_ref"].min(), o["data_ref"].max(), freq="MS")) == len(o)
    checar("dim_enso: ONI contíguo e sem nulos", contiguo and o["oni"].notna().all(),
           "as contagens de meses seguidos exigem série sem buracos")
    d = pd.concat([o[["data_ref", "oni", "fase_enso", "intensidade_enso"]],
                   flags_causais(o["oni"].astype("float64"))], axis=1)
    fase = d["fase_enso"].astype(object)
    eh_ep = fase.isin(["el_nino", "la_nina"])
    bloco = (fase != fase.shift()).cumsum()
    d["episodio_id"] = bloco[eh_ep].rank(method="dense")      # nulo fora de episódio (alinha pelo índice)
    d["mes_no_episodio"] = (d.groupby(bloco).cumcount() + 1).where(eh_ep)
    return aplicar_schema(d, "gold_dim_enso")


def montar_dim_uf(dim_uf, energia):
    base = energia[energia["data_ref"].dt.year.between(*BASE_ANOS)]
    por_uf = base.groupby("cod_ibge_uf")["consumo_mwh"].sum().rename("consumo_base_mwh").reset_index()
    por_sub = base.groupby(["cod_ibge_uf", "subsistema"], as_index=False)["consumo_mwh"].sum()
    pred = (por_sub.loc[por_sub.groupby("cod_ibge_uf")["consumo_mwh"].idxmax(), ["cod_ibge_uf", "subsistema"]]
            .rename(columns={"subsistema": "subsistema_predominante"}))
    n_sub = (energia.groupby("cod_ibge_uf")["subsistema"].nunique().rename("n_subsistemas").reset_index())
    d = (dim_uf.merge(por_uf, on="cod_ibge_uf", how="left", validate="1:1")
               .merge(pred, on="cod_ibge_uf", how="left", validate="1:1")
               .merge(n_sub, on="cod_ibge_uf", how="left", validate="1:1"))
    d["peso_br"] = d["consumo_base_mwh"] / d["consumo_base_mwh"].sum()
    d["peso_regiao"] = d["consumo_base_mwh"] / d.groupby("regiao")["consumo_base_mwh"].transform("sum")
    checar("dim_uf: todas as UFs com consumo no período-base", d["consumo_base_mwh"].notna().all()
           and (d["consumo_base_mwh"] > 0).all(), f"{int(d['consumo_base_mwh'].isna().sum())} sem consumo")
    checar("dim_uf: pesos somam 1 (Brasil e por região)",
           np.isclose(d["peso_br"].sum(), 1.0) and np.allclose(d.groupby("regiao")["peso_regiao"].sum(), 1.0))
    return aplicar_schema(d, "gold_dim_uf")

## 6. Fato de consumo

Soma **cativo + livre** e todos os subsistemas da UF (uma UF pode ter linhas no SIN e em sistema isolado). Estornos negativos entram na soma; o grão que ficar com consumo ≤ 0 é marcado em vez de corrigido, e o alvo em log fica nulo ali.

In [ ]:
def montar_fato_consumo(energia, dim_uf_gold):
    e = energia.copy()
    e["consumo_mwh_cativo"] = e["consumo_mwh"].where(e["tipo_consumidor"] == "cativo", 0.0)
    e["consumo_mwh_livre"] = e["consumo_mwh"].where(e["tipo_consumidor"] == "livre", 0.0)
    e["_neg"] = e["consumo_mwh"] < 0
    f = e.groupby(["cod_ibge_uf", "data_ref", "classe"], as_index=False).agg(
        consumo_mwh=("consumo_mwh", "sum"), consumo_mwh_cativo=("consumo_mwh_cativo", "sum"),
        consumo_mwh_livre=("consumo_mwh_livre", "sum"), n_consumidores=("n_consumidores", "sum"),
        tem_estorno=("_neg", "any"))
    f = f.merge(dim_uf_gold[["cod_ibge_uf", "uf", "regiao"]], on="cod_ibge_uf", how="left", validate="m:1")
    f["ano"] = f["data_ref"].dt.year
    f["mes"] = f["data_ref"].dt.month
    dias = f["data_ref"].dt.days_in_month
    f["consumo_mwh_por_dia"] = f["consumo_mwh"] / dias
    f["consumo_mwh_por_consumidor"] = f["consumo_mwh"] / f["n_consumidores"].astype("float64").where(f["n_consumidores"] > 0)
    f["pct_livre"] = f["consumo_mwh_livre"] / f["consumo_mwh"].where(f["consumo_mwh"] > 0)
    f["flag_consumo_nao_positivo"] = f["consumo_mwh"] <= 0
    return aplicar_schema(f, "gold_fato_consumo_uf_mes")

## 7. Clima: normais, anomalias e agregados

A **normal** de cada UF e mês do ano é a média do período-base (`BASE_ANOS`). A anomalia é a diferença para a normal. Como o período-base fica no início da série, ele cai dentro do treino de qualquer divisão temporal cronológica; a anomalia do teste nunca usa o teste para se calibrar.

Os agregados por região e Brasil usam os **pesos fixos** de `gold_dim_uf`.

In [ ]:
def montar_climatologia(clima, dim_uf_gold):
    base = clima[clima["ano"].between(*BASE_ANOS)]
    c = base.groupby(["cod_ibge_uf", "mes"], as_index=False).agg(
        clim_temp_media_c=("temp_media_c", "mean"), clim_temp_max_media_c=("temp_max_media_c", "mean"),
        clim_precip_total_mm=("precip_total_mm", "mean"), clim_dias_calor=("dias_calor", "mean"),
        n_anos_base=("ano", "nunique"))
    c = c.merge(dim_uf_gold[["cod_ibge_uf", "uf"]], on="cod_ibge_uf", how="left", validate="m:1")
    anos = BASE_ANOS[1] - BASE_ANOS[0] + 1
    checar("climatologia: 27 UFs × 12 meses, todos com o período-base completo",
           len(c) == 27 * 12 and (c["n_anos_base"] == anos).all(),
           f"{len(c)} células; n_anos_base entre {int(c['n_anos_base'].min())} e {int(c['n_anos_base'].max())} (esperado {anos})")
    return aplicar_schema(c, "gold_climatologia_uf")


def montar_clima_uf(clima, climatologia):
    d = clima.merge(climatologia.drop(columns=["uf", "n_anos_base"]), on=["cod_ibge_uf", "mes"],
                    how="left", validate="m:1")
    d["anom_temp_media_c"] = d["temp_media_c"] - d["clim_temp_media_c"]
    d["anom_temp_max_media_c"] = d["temp_max_media_c"] - d["clim_temp_max_media_c"]
    d["anom_precip_mm"] = d["precip_total_mm"] - d["clim_precip_total_mm"]
    d["anom_dias_calor"] = d["dias_calor"].astype("float64") - d["clim_dias_calor"]
    out = aplicar_schema(d, "gold_clima_uf_mes")
    emb = out[out["ano"].between(*BASE_ANOS)].groupby(["cod_ibge_uf", "mes"])["anom_temp_media_c"].mean().abs().max()
    checar("clima_uf: anomalias têm média ~0 no período-base", emb < 1e-9, f"|média| máxima = {emb:.2e}")
    return out


def montar_clima_agregado(clima_uf, dim_uf_gold):
    d = clima_uf.merge(dim_uf_gold[["cod_ibge_uf", "regiao", "peso_br", "peso_regiao"]],
                       on="cod_ibge_uf", how="left", validate="m:1")
    reg = media_ponderada(d, "regiao", "peso_regiao", COLS_CLIMA_NIVEL).rename(columns={"regiao": "id_escopo"})
    reg["escopo"] = "regiao"
    d["brasil"] = "brasil"
    br = media_ponderada(d, "brasil", "peso_br", COLS_CLIMA_NIVEL).rename(columns={"brasil": "id_escopo"})
    br["escopo"] = "brasil"
    out = pd.concat([reg, br], ignore_index=True)
    out["ano"] = out["data_ref"].dt.year
    out["mes"] = out["data_ref"].dt.month
    out = out.rename(columns={"dias_chuva": "dias_chuva_pond", "dias_calor": "dias_calor_pond"})
    return aplicar_schema(out, "gold_clima_agregado_mes")

## 8. Features

Uma linha por UF × mês × classe. Tudo o que é defasado entra por `com_defasagem` (por data). O que **não** existe aqui de propósito: `fase_enso`/`intensidade_enso` (retrospectivas), divisão treino/teste, qualquer ONI com lag 0 (usa o mês seguinte) e consumo, consumidores ou `pct_livre` do próprio mês (são o alvo ou derivam dele; ficam no fato).

`testar_sem_vazamento` perturba um mês T em toda a silver e reconstrói a gold: nenhuma linha anterior a T pode mudar, e na linha de T só podem mudar os alvos e o clima do próprio mês. Para prever **antes** de o mês começar, use só as defasagens ≥ 2 do ONI e as colunas `*_lag1` do clima (o clima do mês e o lag 1 do ONI e do consumo só são conhecidos ao fim dele).

Alvos: `y_log_consumo_dia` (nível) e `y_yoy_log_consumo_dia` (variação sobre o mesmo mês do ano anterior). As autorregressivas (`ar_*`) ficam separadas por prefixo: um modelo que mede o efeito do ENSO deve ser comparado com e sem elas.

In [ ]:
def montar_features(fato, dim_tempo, dim_enso, clima_uf):
    f = fato.merge(dim_tempo[["data_ref", "dias_no_mes", "dias_uteis", "mes_sin", "mes_cos", "t", "flag_covid"]],
                   on="data_ref", how="left", validate="m:1")
    p = f["consumo_mwh_por_dia"].where(f["consumo_mwh_por_dia"] > 0)
    f["y_log_consumo_dia"] = np.log(p)

    # clima do mês e do mês anterior (o consumo é faturado com ~1 mês de atraso)
    f = f.merge(clima_uf[["cod_ibge_uf", "data_ref"] + COLS_CLIMA_NIVEL], on=["cod_ibge_uf", "data_ref"],
                how="left", validate="m:1")
    f = com_defasagem(f, clima_uf, {c: f"{c}_lag1" for c in COLS_CLIMA_LAG1}, 1, ["cod_ibge_uf"])

    # ENSO: só defasagens >= 1 e indicadores causais
    for k in LAGS_ONI:
        f = com_defasagem(f, dim_enso, {"oni": f"oni_lag{k}"}, k)
    for k in LAGS_FLAG:
        f = com_defasagem(f, dim_enso, {"el_nino_causal": f"el_nino_causal_lag{k}",
                                        "la_nina_causal": f"la_nina_causal_lag{k}"}, k)

    # autorregressivas e variação anual, ambas pela data
    ref = f[["cod_ibge_uf", "classe", "data_ref", "y_log_consumo_dia"]]
    f = com_defasagem(f, ref, {"y_log_consumo_dia": "ar_y_log_lag1"}, 1, ["cod_ibge_uf", "classe"])
    f = com_defasagem(f, ref, {"y_log_consumo_dia": "ar_y_log_lag12"}, 12, ["cod_ibge_uf", "classe"])
    f["y_yoy_log_consumo_dia"] = f["y_log_consumo_dia"] - f["ar_y_log_lag12"]
    f = com_defasagem(f, fato, {"pct_livre": "pct_livre_lag1"}, 1, ["cod_ibge_uf", "classe"])

    checar("features: mesmo nº de linhas do fato (nenhum join multiplicou)", len(f) == len(fato), f"{len(f)} vs {len(fato)}")
    return aplicar_schema(f, "gold_features_uf_mes")


def testar_lags_nas_features(feat, dim_enso, clima_uf, amostra=300):
    """Confere os lags com uma conta independente do merge: busca por dicionário, linha a linha."""
    oni = dict(zip(dim_enso["data_ref"], dim_enso["oni"]))
    el = dict(zip(dim_enso["data_ref"], dim_enso["el_nino_causal"]))
    tmp = {(u, d): v for u, d, v in zip(clima_uf["cod_ibge_uf"], clima_uf["data_ref"], clima_uf["temp_media_c"])}
    linhas = feat.sample(min(amostra, len(feat)), random_state=1)
    for _, r in linhas.iterrows():
        for k in LAGS_ONI:
            antes = r["data_ref"] - pd.DateOffset(months=k)
            assert np.isclose(r[f"oni_lag{k}"], oni[antes]), (r["data_ref"], k)
        for k in LAGS_FLAG:
            antes = r["data_ref"] - pd.DateOffset(months=k)
            assert bool(r[f"el_nino_causal_lag{k}"]) == bool(el[antes]), (r["data_ref"], k)
        antes = r["data_ref"] - pd.DateOffset(months=1)
        esperado = tmp.get((r["cod_ibge_uf"], antes), np.nan)
        got = r["temp_media_c_lag1"]
        assert (pd.isna(esperado) and pd.isna(got)) or np.isclose(got, esperado), (r["cod_ibge_uf"], r["data_ref"])
    checar("features: lags de ONI, flags e clima conferem com busca independente", True, f"{len(linhas)} linhas amostradas")


def testar_sem_vazamento(s, ini, fim, feat):
    """Perturba o mês T em toda a silver (consumo cativo, consumidores, clima, ONI) e reconstrói a gold.
    Antes de T nada pode mudar (nada olha o futuro). Em T só mudam os alvos e o clima do próprio mês
    (exógeno): nenhuma feature usa o consumo do mês. Em T+1 a perturbação tem de aparecer nos lags,
    senão o teste não estaria perturbando nada."""
    T = fim - pd.DateOffset(months=24)
    p = {n: df.copy() for n, df in s.items()}
    e = p["silver_energia_uf"]
    m = e["data_ref"] == T
    e.loc[m & (e["tipo_consumidor"] == "cativo"), "consumo_mwh"] *= 1.7   # só o cativo: muda também pct_livre
    e.loc[m, "n_consumidores"] += 1000
    c = p["silver_clima_uf_mes"]
    c.loc[c["data_ref"] == T, ["temp_media_c", "temp_max_media_c", "precip_total_mm"]] += 5.0
    o = p["silver_oni_mes"]
    o.loc[o["data_ref"] == T, "oni"] += 1.0

    n = len(VALIDACOES)
    with contextlib.redirect_stdout(io.StringIO()):
        outra = montar_tabelas(p, ini, fim)["gold_features_uf_mes"]
    del VALIDACOES[n:]

    def mudaram(mes):
        a, b = feat[feat["data_ref"] == mes], outra[outra["data_ref"] == mes]
        return {col for col in feat.columns if not a[col].equals(b[col])}

    antes = {col for col in feat.columns
             if not feat.loc[feat["data_ref"] < T, col].equals(outra.loc[outra["data_ref"] < T, col])}
    em_t, depois = mudaram(T), mudaram(T + pd.DateOffset(months=1))
    checar("anti-vazamento: perturbar o mês T não muda nenhuma linha anterior", not antes, sorted(antes))
    checar("anti-vazamento: em T só mudam alvos e clima do próprio mês",
           "y_log_consumo_dia" in em_t and em_t <= set(COLS_ALVO + COLS_CLIMA_NIVEL), sorted(em_t))
    sensores = {"ar_y_log_lag1", "pct_livre_lag1", "oni_lag1", "temp_media_c_lag1"}
    checar("anti-vazamento: a perturbação aparece nos lags de T+1 (o teste é sensível)",
           sensores <= depois, sorted(sensores - depois))


## 9. Orquestração, gravação e relatório

Gravação atômica (arquivo temporário + `os.replace`), como na silver. Cada execução registra no relatório os hashes das tabelas **de entrada**, para rastrear de qual silver cada gold saiu.

In [ ]:
def informativos(s, fato, feat, clima_uf):
    """Números para o relatório e avisos que não devem travar a execução."""
    energia = s["silver_energia_uf"]
    checar("fato: consumo total igual ao da silver", np.isclose(fato["consumo_mwh"].sum(), energia["consumo_mwh"].sum(), rtol=1e-9),
           f"{fato['consumo_mwh'].sum():,.2f} vs {energia['consumo_mwh'].sum():,.2f}")
    ufs_mes = fato[["cod_ibge_uf", "data_ref"]].drop_duplicates()
    checar("fato: todo UF-mês da energia presente", len(ufs_mes) == 27 * fato["data_ref"].nunique(), f"{len(ufs_mes)} UF-meses")
    esperado_grade = 27 * fato["data_ref"].nunique() * fato["classe"].nunique()
    checar("fato: grade UF × mês × classe completa", len(fato) == esperado_grade,
           f"{len(fato)} linhas de {esperado_grade} possíveis (faltantes não viram zero)", critico=False)
    checar("fato: sem consumo não positivo após a soma", not fato["flag_consumo_nao_positivo"].any(),
           f"{int(fato['flag_consumo_nao_positivo'].sum())} grãos (alvo em log fica nulo)", critico=False)
    fora = fato["pct_livre"].dropna()
    checar("fato: pct_livre dentro de [0, 1]", bool(fora.between(0, 1).all()),
           f"{int((~fora.between(0, 1)).sum())} fora (estornos)", critico=False)

    # nulos esperados nas features: só onde falta histórico
    ini = feat["data_ref"].min()
    nulos = {}
    for c in COLS_ONI_LAGS + COLS_FLAG_LAGS:
        nulos[c] = int(feat[c].isna().sum())
    checar("features: ONI e flags defasados sem nulos", sum(nulos.values()) == 0, nulos)
    primeiro = feat["data_ref"] == ini
    n1 = int(feat["temp_media_c_lag1"].isna().sum())
    checar("features: clima lag 1 nulo só no primeiro mês", n1 == int(primeiro.sum()), f"{n1} nulos; primeiro mês tem {int(primeiro.sum())} linhas")
    ate12 = feat["data_ref"] < ini + pd.DateOffset(months=12)
    validos = set(zip(feat.loc[feat["y_log_consumo_dia"].notna(), "cod_ibge_uf"],
                      feat.loc[feat["y_log_consumo_dia"].notna(), "classe"],
                      feat.loc[feat["y_log_consumo_dia"].notna(), "data_ref"]))
    resto = feat[~ate12]
    doze_antes = resto["data_ref"].dt.to_period("M").sub(12).dt.to_timestamp()
    esperado12 = sum((u, c, d) not in validos for u, c, d in zip(resto["cod_ibge_uf"], resto["classe"], doze_antes))
    n12 = int(resto["ar_y_log_lag12"].isna().sum())
    checar("features: ar_y_log_lag12 nulo só onde o mês de 12 meses antes falta ou tem consumo <= 0",
           n12 == esperado12, f"{n12} nulos; esperado {esperado12}")

    pouca_variacao = (clima_uf.assign(zero=clima_uf["dias_calor"] == 0).groupby("uf")["zero"].mean())
    return {
        "janela": [str(ini.date()), str(feat["data_ref"].max().date())],
        "ufs_dias_calor_quase_sempre_zero": sorted(pouca_variacao[pouca_variacao >= 0.95].index.tolist()),
        "consumo_nao_positivo": int(fato["flag_consumo_nao_positivo"].sum()),
        "nulos_features": {c: int(v) for c, v in feat.isna().sum().items() if v},
    }


def montar_tabelas(s, ini, fim):
    energia, clima, oni = s["silver_energia_uf"], s["silver_clima_uf_mes"], s["silver_oni_mes"]
    dim_tempo = montar_dim_tempo(ini, fim)
    dim_enso = montar_dim_enso(oni)
    dim_uf = montar_dim_uf(s["dim_uf"], energia)
    fato = montar_fato_consumo(energia, dim_uf)
    climatologia = montar_climatologia(clima, dim_uf)
    clima_uf = montar_clima_uf(clima, climatologia)
    clima_agr = montar_clima_agregado(clima_uf, dim_uf)
    feat = montar_features(fato, dim_tempo, dim_enso, clima_uf)
    return {"gold_dim_tempo": dim_tempo, "gold_dim_enso": dim_enso, "gold_dim_uf": dim_uf,
            "gold_fato_consumo_uf_mes": fato, "gold_climatologia_uf": climatologia,
            "gold_clima_uf_mes": clima_uf, "gold_clima_agregado_mes": clima_agr,
            "gold_features_uf_mes": feat}


def construir_gold():
    VALIDACOES.clear()
    testar_defasagem_por_data()
    testar_flags_causais()
    testar_dias_uteis()

    s = ler_silver()
    ini, fim = conferir_entrada(s)
    tabelas = montar_tabelas(s, ini, fim)
    fato, feat, clima_uf = (tabelas[n] for n in ("gold_fato_consumo_uf_mes", "gold_features_uf_mes", "gold_clima_uf_mes"))
    testar_lags_nas_features(feat, tabelas["gold_dim_enso"], clima_uf)
    testar_sem_vazamento(s, ini, fim, feat)
    info = informativos(s, fato, feat, clima_uf)
    rel = {
        "gerado_em": datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"),
        "parametros": {"BASE_ANOS": list(BASE_ANOS), "LAGS_ONI": list(LAGS_ONI), "LAGS_FLAG": list(LAGS_FLAG),
                       "LIMIAR_ENSO": LIMIAR_ENSO, "MESES_EPISODIO": MESES_EPISODIO,
                       "COVID": [str(COVID_INI.date()), str(COVID_FIM.date())]},
        "entradas_silver_hash": {n: hash_tabela(df) for n, df in s.items()},
        "data_coleta": {"epe": data_coleta(s["silver_energia_uf"]["_id_carga"]),
                        "openmeteo": data_coleta(s["silver_clima_uf_mes"]["_id_carga"]),
                        "noaa": data_coleta(s["silver_oni_mes"]["_id_carga"])},
        "linhas": {n: int(len(df)) for n, df in tabelas.items()},
        "informativos": info,
        "validacoes": VALIDACOES.copy(),
    }
    return tabelas, rel


def gravar_gold(tabelas, rel):
    """Atômico: escreve em temporário e troca com os.replace."""
    GOLD.mkdir(parents=True, exist_ok=True)
    RELATORIOS.mkdir(parents=True, exist_ok=True)
    for nome, df in tabelas.items():
        destino = GOLD / f"{nome}.parquet"
        tmp = destino.with_suffix(".parquet.tmp")
        df.to_parquet(tmp, index=False)
        os.replace(tmp, destino)
    rel["hash_gold"] = hashes_em_disco()
    (RELATORIOS / f"gold_{rel['gerado_em']}.json").write_text(json.dumps(rel, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    dicionario(rel["data_coleta"]).to_csv(GOLD / "dicionario_gold.csv", index=False, encoding="utf-8")


def hashes_em_disco():
    return {p.stem: hash_tabela(pd.read_parquet(p)) for p in sorted(GOLD.glob("*.parquet"))}


def executar_gold():
    tabelas, rel = construir_gold()
    gravar_gold(tabelas, rel)
    return tabelas, rel

## 10. Execução

In [ ]:
tabelas, rel = executar_gold()
print()
print(pd.Series(rel["linhas"], name="linhas").to_frame())
print("\nJanela:", rel["informativos"]["janela"])
print("UFs com dias_calor ~sempre 0 (decidir se a métrica serve):", rel["informativos"]["ufs_dias_calor_quase_sempre_zero"])

## 11. Idempotência

Reconstruir a gold a partir da mesma silver tem de produzir exatamente as mesmas tabelas (hash do conteúdo, independente da ordem das linhas).

In [ ]:
antes = hashes_em_disco()
tabelas2, rel2 = executar_gold()
depois = hashes_em_disco()
iguais = {n: antes[n] == depois[n] for n in antes}
checar("idempotência: segunda execução gera as mesmas tabelas", all(iguais.values()),
       {n: ok for n, ok in iguais.items() if not ok} or "8 tabelas idênticas")